In [1]:
import geopandas as gpd
import pandas as pd
import yaml
from pathlib import Path
from tqdm.auto import tqdm
from rasterio.crs import CRS

# Events

In [2]:
event_ymls = list(Path('db/events/').glob('*.yml'))
event_ymls[:2]

[PosixPath('db/events/porto_alegre_flood_2024.yml'),
 PosixPath('db/events/chiapas_fire_2024.yml')]

In [3]:
KEYS = ['event_name',
        'event_date',
        'mgrs_tiles',
        'source_id',
        'links']
def open_one_event(path: str) -> dict:
    with open(path) as file:
        nested_data = yaml.safe_load(file)
    data = nested_data['event']
    out = {}
    keys_not_present = [key for key in KEYS if key not in data]
    if keys_not_present:
        print(f'{path} does not have {", ".join(keys_not_present)}')
        print('skipping...')
        return {}
    out = {key: data[key] for key in KEYS}
    return out

In [4]:
event_records = list(map(open_one_event, event_ymls))
event_records[0]

{'event_name': 'porto_alegre_flood_2024',
 'event_date': '2024-05-28',
 'mgrs_tiles': ['22JDM', '22JDN'],
 'source_id': 'EMSN194_STD_AOI01_P04FLDEL01_FloodExtent_v01',
 'links': ['https://mapping.emergency.copernicus.eu/activations/EMSN194/']}

In [5]:
df_event = pd.DataFrame(event_records)
df_event.head()

,event_name,event_date,mgrs_tiles,source_id,links
0,porto_alegre_flood_2024,2024-05-28,"[22JDM, 22JDN]",EMSN194_STD_AOI01_P04FLDEL01_FloodExtent_v01,[https://mapping.emergency.copernicus.eu/activ...
1,chiapas_fire_2024,2024-03-24,[15QUU],Copernicus EMSR717,[data: https://rapidmapping.emergency.copernic...
2,yajiang_fire_2024,2024-03-15,"[47RQP, 47RPP]",UNOSAT via humanitarian data exchange,[https://data.humdata.org/dataset/the-wildfire...
3,bangladesh_coastal_flood_2024,2024-07-08,"[45QYE, 45QYF, 45QZE, 45QZF]",UNOSAT via humanitarian data exchange ST1_2024...,[https://data.humdata.org/dataset/satellite-de...
4,park_fire_2024,2024-07-24,"[10TEK, 10TFK]",WFIGS Park Fire 2024,[https://en.wikipedia.org/wiki/Park_Fire]


In [6]:
df_event.to_parquet('event_metadata.parquet')

# Event Perimeters

In [7]:
def open_one_perimeter(event_name: str):
    perimeter_dir = Path('db/event_perimeters/')
    path_parquet = perimeter_dir / f'{event_name}.parquet'
    if not path_parquet.exists():
        print(f'{path_parquet} does not exist')
        print('skipping...')
        return {}
    df_geo = gpd.read_parquet(path_parquet)
    geo = df_geo.geometry.union_all()
    return {'event_name': event_name, 'geometry': geo}

In [8]:
event_names = df_event['event_name'].tolist()
perimeter_data = [open_one_perimeter(event) for event in tqdm(event_names)]

  0%|          | 0/25 [00:00<?, ?it/s]

In [9]:
df_perimeters = gpd.GeoDataFrame(perimeter_data,
                                 crs=CRS.from_epsg(4326))
df_perimeters.head()

,event_name,geometry
0,porto_alegre_flood_2024,"MULTIPOLYGON (((-51.26193 -30.11187, -51.26189..."
1,chiapas_fire_2024,"MULTIPOLYGON (((-94.53702 16.47015, -94.53692 ..."
2,yajiang_fire_2024,"MULTIPOLYGON (((101.0631 30.05153, 101.06069 3..."
3,bangladesh_coastal_flood_2024,GEOMETRYCOLLECTION (POLYGON ((88.98224 22.4023...
4,park_fire_2024,"MULTIPOLYGON (((-121.93048 39.97492, -121.9296..."


In [10]:
df_perimeters.to_parquet('event_perimeters.parquet', compression='zstd')

# Event Sites

In [11]:
df_event_sites = df_perimeters.copy()
df_event_sites.geometry = df_perimeters.geometry.centroid

/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_75136/2105504291.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df_event_sites.geometry = df_perimeters.geometry.centroid


In [12]:
df_event_sites.to_parquet('event_sites.parquet')

# Event Extents

In [13]:
def open_one_extent(event_name: str):
    extent_dir = Path('db/event_extents/')
    path_parquet = extent_dir / f'{event_name}.parquet'
    if not path_parquet.exists():
        print(f'{path_parquet} does not exist')
        print('skipping...')
        return {}
    
    df_geo = gpd.read_parquet(path_parquet)
    if df_geo.crs != CRS.from_epsg(4326):
        print(f'CRS for {event_name} is not lon/lat')
        print(f'Please reproject; skipping...')
        return {}
    geo = df_geo.geometry.union_all()
    return {'event_name': event_name, 'geometry': geo}

In [14]:
event_names = df_event['event_name'].tolist()
extent_data = [open_one_extent(event) for event in tqdm(event_names)]

  0%|          | 0/25 [00:00<?, ?it/s]

In [15]:
df_extent = gpd.GeoDataFrame(extent_data, crs=CRS.from_epsg(4326))
df_extent.to_parquet('event_extents.parquet')